## Membership Inference Attack (MIA)

Dalam era privasi data, kebocoran data latih melalui Membership Inference Attack (**MIA**) menjadi ancaman serius bagi model Machine Learning. Serangan ini bertujuan untuk menebak apakah suatu sampel dokumen tertentu digunakan dalam proses pelatihan (**Member**) atau tidak (**Non-Member**). Di soal ini, sebuah **Target Model** telah dilatih menggunakan dataset dokumen teks yang berisi kata-kata sensitif. Karena model dilatih agar overfit dengan kata-kata tersebut, model menunjukkan tingkat sensitivitas yang berbeda ketika kata-kata tertentu dimodifikasi.

Tugas Anda adalah membuat algoritma yang dapat **memprediksi status keanggotaan (Membership) dari serangkaian dokumen** evaluasi pada Target Model dengan memanfaatkan akses terbatas ke API model target (**Black-box Query**) dan sebuah **Shadow Model** untuk kalibrasi.


## Persiapan


In [1]:
%pip install -q scikit-learn==1.6.1 dill==0.3.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 1.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import warnings

import dill
import numpy as np
import pandas as pd

def cari_folder_data(berkas_wajib):
    """Cari folder dataset pada komputer lokal atau Kaggle."""
    kandidat = [Path("kaggle_data"), Path("../kaggle_data"), Path(".")]
    folder_kaggle = Path("/kaggle/input")
    if folder_kaggle.exists():
        kandidat.extend(path.parent for path in folder_kaggle.rglob(berkas_wajib))
    for folder in kandidat:
        if (folder / berkas_wajib).exists() and (folder / "sample_submission.csv").exists():
            return folder
    raise FileNotFoundError(f"Dataset Kaggle dengan berkas {berkas_wajib} tidak ditemukan")


JALUR_SUBMISSION = (
    Path("/kaggle/working/submission.csv")
    if Path("/kaggle/working").exists()
    else Path("submission.csv")
)

DATA_DIR = cari_folder_data("challenge_environment.pkl")
print("Folder data:", DATA_DIR.resolve())


Folder data: /kaggle/input/competitions/soal-1-1-final-ekka-2026/kaggle_data


## Format Data & API

Anda diberikan sebuah environment yang telah di-load dari file `.pkl` dengan komponen sebagai berikut:



1.   `eval_texts` (list of strings): Daftar 2.000 dokumen teks evaluasi yang perlu diprediksi status keanggotaannya.
2.    `query_fn` (fungsi / callable): Fungsi API untuk query ke **Target Model**. Menerima list teks dan mengembalikan tensor probabilitas berbentuk (N, 2) untuk target model, yaitu [P(class 0), P(class 1)], dengan 0 adalah non-member dan 1 adalah member

  *   **input**: `list` berisi teks
  *   **output**: `numpy.ndarray` berukuran `(N, 2)` yang mengembalikan probabilitas kelas `[P(class 0), P(class 1)]`

3.    `shadow_data` (dictionary): Data dan fungsi Shadow Model yang dilatih pada arsitektur serupa untuk membantu kalibrasi/pembelajaran fitur serangan:

  *   `texts`: `list` dokumen teks shadow (`100 member` + `100 non-member`)
  *   `labels`: Array biner status membership shadow (1 untuk Member, 0 untuk Non-Member).
  *   `query_fn`: Fungsi query API untuk Shadow Model (dengan spesifikasi argumen yang sama dengan `query_fn` untuk Target Model).


In [2]:
def rebuild_query_fn(serialized_function):
      objects = [
          cell.cell_contents
          for cell in serialized_function.__closure__
      ]

      model = next(obj for obj in objects if hasattr(obj, "predict_proba"))
      vectorizer = next(obj for obj in objects if hasattr(obj, "vocabulary_"))

      def safe_query_fn(texts):
          features = vectorizer.transform(texts)
          return model.predict_proba(features)

      return safe_query_fn

In [3]:
with open(DATA_DIR / "challenge_environment.pkl", "rb") as file:
  data = dill.load(file)

eval_texts = data["eval_texts"]
shadow_data = data["shadow_data"]

query_fn = rebuild_query_fn(data["query_fn"])
shadow_data["query_fn"] = rebuild_query_fn(shadow_data["query_fn"])
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if len(eval_texts) != len(test) or sample_submission["id"].tolist() != test["id"].tolist():
    raise ValueError("Urutan atau jumlah ID tidak cocok dengan eval_texts")


In [4]:
len(eval_texts)


2000

In [5]:
eval_texts[:5]


['log key password key user record',
 'science system input method structure data feature method',
 'function data learning science variable method network analysis input',
 'output variable function python performance input function variable top secret access key',
 'science variable method structure output structure method']

In [6]:
shadow_texts = shadow_data['texts']
shadow_labels = shadow_data['labels']
shadow_query_fn = shadow_data['query_fn']


In [7]:
def ngram(txt):
    ls = []
    words = txt.split()
    for i in range(len(words)):
        ls.append(words[i])
    for i in range(len(words)-1):
        ls.append(f"{words[i]} {words[i+1]}")
    for i in range(len(words)-2):
        ls.append(f"{words[i]} {words[i+1]} {words[i+2]}")
    return sorted(list(set(ls)))

In [8]:
tottext = shadow_texts+eval_texts
vocab = []
for sentence in tottext:
    dic = ngram(sentence.lower())
    for word in dic:
        vocab.append(word)

In [9]:
tf = []
for doc in tottext:
    freq = []
    wordindoc = doc.split()
    wordlength = len(wordindoc)
    for word in vocab:
        wordfreq = 0
        for docword in wordindoc:
            if docword == word:
                wordfreq+=1
        freq.append(wordfreq)
    freq = np.asarray(freq)
    tf.append(freq/wordlength)

In [10]:
tf = np.asarray(tf)
tf.shape

(2200, 49559)

In [11]:
idf = []
for word in vocab:
    cnt = 0
    for doc in tottext:
        if word in doc:
            cnt+=1
    idf.append(np.log(2200/cnt+1))
idf = np.asarray(idf)

In [12]:
idf.shape

(49559,)

In [13]:
tfidf = tf * idf

## Tugas Anda

Buatlah fungsi prediksi yang menerima `eval_texts`, `query_fn`, dan `shadow_data` (lihat contoh kode baseline di bawah), kemudian mengembalikan:

1.   `predictions`: Array biner `0` atau `1` berukuran `(2000,)` sebagai tebakan status keanggotaan dokumen (`1` = Member, `0` = Non-Member).
2.   `scores`: Array skor confidence/kontinuitas berukuran `(2000,)` untuk evaluasi metrik ROC-AUC. Skor-nya berkisar dari `0` hingga `1`.


## Metode Baseline: Prediction Entropy Attack

Mengeksploitasi tingkat keyakinan model (confidence) berdasarkan asumsi bahwa model lebih yakin (overconfident) pada data latih (Member) dibanding data baru (Non-Member).

Cara Kerja:
1.   Query Model: Mengambil probabilitas prediksi $P(y\vert{}x)$ untuk setiap dokumen.
2.   Hitung Entropi: Mengukur ketidakpastian prediksi dengan rumus Shannon Entropy: $H(X) = -\sum P(x) \log P(x)$; Entropi rendah $\rightarrow$ Model sangat yakin; Entropi tinggi $\rightarrow$ Model ragu-ragu.
3.   Thresholding: Menentukan batas pemisah menggunakan nilai tengah (median) dari seluruh entropi.
4.   Klasifikasi: Entropi < Median $\rightarrow$ Member (Label `1`); Entropi $\ge$ Median $\rightarrow$ Non-Member (Label `0`).


In [14]:
def baseline_membership_inference(eval_texts, query_fn, shadow_data): #count_nonzero axis=0 tf 1+n / 1+df + 1
    # Untuk baseline, kita tidak menggunakan shadow_data
    probs = np.asarray(query_fn(eval_texts), dtype=float)
    if probs.shape != (len(eval_texts), 2):
        raise ValueError("query_fn harus menghasilkan array berbentuk (N, 2)")

    eps = 1e-12
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    threshold = np.median(entropy)
    predictions = (entropy < threshold).astype(int)

    # Versi asli memakai -entropy. Untuk submission Kaggle, skor yang urutannya
    # sama dipetakan ke rentang 0 sampai 1.
    scores = np.clip(1.0 - entropy / np.log(2.0), 0.0, 1.0)
    return predictions, scores


## Membuat `submission.csv`

Kaggle hanya menilai skor kontinu pada kolom `membership_score`. Prediksi biner
dapat digunakan untuk analisis lokal, tetapi tidak perlu dimasukkan ke file.


In [15]:
predictions, scores = baseline_membership_inference(
    eval_texts, query_fn, shadow_data
)
submission = sample_submission.copy()
submission["membership_score"] = scores
if not np.isfinite(submission["membership_score"]).all():
    raise ValueError("Submission mengandung nilai non-finite")
if not submission["membership_score"].between(0.0, 1.0).all():
    raise ValueError("membership_score harus berada pada rentang 0 sampai 1")
submission.to_csv(JALUR_SUBMISSION, index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
print("Prediksi Member:", int(predictions.sum()), "dari", len(predictions))
display(submission.head())


Submission ditulis ke: /kaggle/working/submission.csv
Prediksi Member: 1000 dari 2000


,id,membership_score
0,eval_0000,0.856383
1,eval_0001,0.976591
2,eval_0002,0.983241
3,eval_0003,0.915403
4,eval_0004,0.963617


In [78]:
JALUR_SUBMISSION = (
    Path("/kaggle/working/submission2.csv")
    if Path("/kaggle/working").exists()
    else Path("submission2.csv")
)
scores = [0.5] * 2000
submission = sample_submission.copy()
submission["membership_score"] = scores
if not np.isfinite(submission["membership_score"]).all():
    raise ValueError("Submission mengandung nilai non-finite")
if not submission["membership_score"].between(0.0, 1.0).all():
    raise ValueError("membership_score harus berada pada rentang 0 sampai 1")
submission.to_csv(JALUR_SUBMISSION, index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
print("Prediksi Member:", int(predictions.sum()), "dari", len(predictions))
display(submission.head())

Submission ditulis ke: /kaggle/working/submission2.csv
Prediksi Member: 1000 dari 2000


,id,membership_score
0,eval_0000,0.5
1,eval_0001,0.5
2,eval_0002,0.5
3,eval_0003,0.5
4,eval_0004,0.5


In [16]:
class knnr:
    def __init__(self, k=3):
        self.k = k
    def fit(self, x,y):
        self.x=np.asarray(x)
        self.y=np.asarray(y)
    def predict(self, x):
        return [self.pred_one(xa) for xa in x]
    def pred_one(self, x):
        dis = np.sqrt(np.sum((self.x-x)**2, axis=1))
        indices = np.argsort(dis)[:self.k]
        label = self.y[indices]
        return np.mean(label.reshape(-1))

In [17]:
model = knnr()
model.fit(x=tfidf[:200], y = shadow_labels)

In [19]:
scores = model.predict(tfidf[200:])
JALUR_SUBMISSION = (
    Path("/kaggle/working/submission8.csv")
    if Path("/kaggle/working").exists()
    else Path("submission8.csv")
)
submission = sample_submission.copy()
submission["membership_score"] = scores
if not np.isfinite(submission["membership_score"]).all():
    raise ValueError("Submission mengandung nilai non-finite")
if not submission["membership_score"].between(0.0, 1.0).all():
    raise ValueError("membership_score harus berada pada rentang 0 sampai 1")
submission.to_csv(JALUR_SUBMISSION, index=False)

print("Submission ditulis ke:", JALUR_SUBMISSION)
print("Prediksi Member:", int(predictions.sum()), "dari", len(predictions))
display(submission.head())

Submission ditulis ke: /kaggle/working/submission8.csv
Prediksi Member: 1000 dari 2000


,id,membership_score
0,eval_0000,0.333333
1,eval_0001,0.666667
2,eval_0002,0.666667
3,eval_0003,0.666667
4,eval_0004,0.666667


## Pemeriksaan akhir

Jalankan **Restart & Run All** dan pastikan `submission.csv` memiliki tepat
2.000 ID serta skor finite antara 0 dan 1. Unggah hanya `submission.csv` ke
halaman kompetisi Kaggle.
